# exp044_stratified_groupkfold_cv_audit train

Diagnostic notebook for stratified well GroupKFold fold balance and OOF stress reporting.

## Contents

1. Setup and configuration
2. Build stratified well folds
3. Re-aggregate available OOF predictions
4. Metrics and artifacts


## 1. Setup and configuration


In [ ]:
from __future__ import annotations

import json
import os

import pandas as pd

from settings import EXPERIMENT_NAME, ExperimentPaths, load_config
from stratified_groupkfold_cv_audit import run_audit

DEBUG = os.environ.get("EXPERIMENT_DEBUG", "0") == "1"
MAX_WELLS_ENV = os.environ.get("EXPERIMENT_MAX_WELLS")
MAX_WELLS = int(MAX_WELLS_ENV) if MAX_WELLS_ENV else None

paths = ExperimentPaths()
paths.require_kaggle_runtime()
paths.ensure_output_dirs()
config = load_config()

print("Experiment:", EXPERIMENT_NAME)
print("Train data:", paths.train_data_dir)
print("Artifacts:", paths.artifacts_dir)
print("Metric:", config.get("validation", {}).get("metric"))
print("Debug:", DEBUG, "Max wells:", MAX_WELLS)
print("OOF sources:", [item.get("name") for item in config.get("audit", {}).get("oof_sources", [])])


## 2. Build stratified well folds


In [ ]:
metrics = run_audit(max_wells=MAX_WELLS, allow_missing_oof=True)

folds = pd.read_csv(paths.artifacts_dir / "well_metadata_stratified_folds.csv")
fold_summary = pd.read_csv(paths.artifacts_dir / "fold_balance_summary.csv")
bucket_summary = pd.read_csv(paths.artifacts_dir / "fold_bucket_distribution.csv")

display(folds.head())
display(fold_summary)
display(bucket_summary.head(20))


## 3. Re-aggregate available OOF predictions


In [ ]:
source_status = pd.read_csv(paths.artifacts_dir / "oof_source_status.csv")
display(source_status)

segment_path = paths.artifacts_dir / "stratified_oof_segment_metrics.csv"
if segment_path.exists():
    segment_metrics = pd.read_csv(segment_path)
    display(segment_metrics[segment_metrics["segment_type"] == "overall"])
    display(segment_metrics.head(30))
else:
    print("No OOF segment metrics were generated; configured OOF sources were unavailable.")


## 4. Metrics and artifacts


In [ ]:
print(json.dumps(metrics, indent=2, sort_keys=True)[:4000])
print("Metrics written:", paths.metrics_path)
print("Artifacts written:")
for path in sorted(paths.artifacts_dir.glob("*.csv")):
    print("-", path.name)
